# S1 - Calibration with the beta light: gain, read noise, linearity, sample-plane reference

One attended session per camera at its operating point (`slim.CAMERAS`), about an hour. Repeat it
when the optics change (session 2 with the 50 mm f/0.95 pair): same notebook, new `OPTICS`.

1. **Bias**: shutter closed, shortest exposure. Offset and read noise.
2. **Photon transfer + linearity**: the beta light sits in the **infinity space** with a lens in
   front of it, so the camera-side lens images it in focus onto the sensor. Its brightness profile
   gives a ramp of grey values in every frame; an exposure ladder stretches the ramp. At every
   exposure: `N_FRAMES` frames with the shutter open, `N_FRAMES` closed. The fit splits the frames
   into halves (bins on one half, measures on the other), so noise cannot bias the slope.
   Result: ADU per e- (for the EMCCD the slope is divided by F² = 2), read noise in e-,
   linearity residuals, dark current over the short ladder.
3. **Sample-plane reference** (optional, `REFERENCE_MS`): the same beta light at the sample plane,
   in focus. Its total signal in e-/s measures what the **whole relay** collects, so session 1
   (4x/0.20 pair) and session 2 (50/0.95 pair) compare directly. The infinity-space ramp bypasses
   the sample-side lens and cannot do that.

The shutter has to sit between the relay and the sensor, so it also blocks the beta light.

Runs in the foreground: when the beta light has to move, the cell waits for Enter.
`SIMULATE = True` runs everything against the simulator; `DRY_RUN = True` takes 3-frame stacks on
the real camera without waiting for the cooler.

## 1 Connect

In [ ]:
from pathlib import Path
import slim
bp = slim.bp

SIMULATE = True        # True: slim.Simulator stands in for ImSwitch, camera, shutter, pump, beta light
CAMERA = "toupcam"     # "toupcam" or "andor": the one operating point of slim.CAMERAS
# None = $IMSWITCH_API_URL (ImSwitch's own notebooks). Andor rig e.g. "http://192.168.178.50", port 80
IMSWITCH_URL, PORT = "http://localhost", 8001

client = slim.Simulator(CAMERA) if SIMULATE else bp.connect(IMSWITCH_URL, PORT)
DATA_ROOT = slim.DATA_ROOT / "simulated" if SIMULATE else slim.DATA_ROOT
print("Connected to", client.base_uri, "| detectors:", client.settingsManager.getDetectorNames())
print("Operating point:", slim.CAMERAS[CAMERA]["settings"])

## 2 Settings

In [ ]:
RUN_NAME = f"{CAMERA}_calibration"
DRY_RUN = False
OPTICS = {"name": "4x/0.20 pair", "magnification": 1.0}   # session 2: {"name": "50/0.95 pair", ...}

# Exposure ladder for the beta light in the infinity space; set the longest with the check below
# (brightest part ~70 % of full scale). Geometric steps over at least a decade.
PTC_EXPOSURES_MS = {"toupcam": [50, 100, 200, 500, 1000, 2000],
                    "andor": [1, 2, 5, 10, 20, 50]}[CAMERA]
N_FRAMES = 8           # light and dark frames per exposure (even: the fit splits them in halves)
N_BIAS = 20
BIAS_MS = 1
REFERENCE_MS = None    # e.g. 10_000: beta light at the sample plane; None skips it
PIXEL_STEP = 2 if CAMERA == "toupcam" else 1   # analysis uses every n-th pixel (9 MP frames)

# The shutter and the pump. Until ImSwitch exposes them, leave them None: the notebook then asks
# you (shutter at the start and end of a run, medium exchange when it is due).
# Once the functions exist, e.g.:
#   SHUTTER_OPEN = slim.endpoint(client, "<Controller>/<function>", {"<param>": 1})
#   SHUTTER_CLOSE = slim.endpoint(client, "<Controller>/<function>", {"<param>": 0})
# or any Python function that moves it.
SHUTTER_OPEN = SHUTTER_CLOSE = None

if SIMULATE:
    PTC_EXPOSURES_MS = {"toupcam": [2, 5, 10, 20, 25], "andor": [5, 10, 20, 50, 100]}[CAMERA]
    REFERENCE_MS, PIXEL_STEP = PTC_EXPOSURES_MS[-1], 1
if DRY_RUN:
    N_FRAMES, N_BIAS = 4, 4

STEPS = slim.calibration_schedule(PTC_EXPOSURES_MS, N_FRAMES, N_BIAS, BIAS_MS, REFERENCE_MS)
print(slim.describe(STEPS))

## 3 Run folder, camera settings, environment log, cooldown

Applies the operating point by name and reads it back, starts the environment log, waits for the
set point and lets the sensor settle (not in a dry run or a simulation).

In [ ]:
run, env, detector, preset = slim.prepare(
    client, CAMERA, "S1_calibration", RUN_NAME, dry_run=DRY_RUN, data_root=DATA_ROOT,
    wait=not (DRY_RUN or SIMULATE),
    kind="slim_calibration", title=f"S1 - calibration, {slim.CAMERAS[CAMERA]['label']}",
    description="Bias, photon transfer and linearity with the beta light in the infinity space, "
                "dark current over the ladder, beta light at the sample plane.",
    optics=OPTICS, steps=STEPS, analysis={"pixel_step": PIXEL_STEP})

if SIMULATE:
    shutter, operator = client.shutter(log=run.log), client.operator
else:
    operator = slim.Operator(log=run.log, dry_run=DRY_RUN)
    shutter = (slim.Shutter("callable", SHUTTER_OPEN, SHUTTER_CLOSE, operator, log=run.log)
               if SHUTTER_OPEN else slim.Shutter("prompt", operator=operator, log=run.log))
print("shutter:", shutter.mode)

## 4 Exposure check

Beta light in the infinity space, shutter open, longest exposure of the ladder. Re-run with a
different `PTC_EXPOSURES_MS` (cell 2, then this one) until the brightest part sits near 70 % of
full scale and nothing clips.

In [ ]:
operator.ask("Put the beta light into the INFINITY SPACE (lens in front of it) and close the enclosure")
shutter.set("open")
client.settingsManager.setDetectorExposureTime(detector, max(PTC_EXPOSURES_MS))
rel = bp.snap_recording(client, f"{run.dir.name}_check")
run.log(f"exposure check at {max(PTC_EXPOSURES_MS)} ms: {rel}")
bp.show_exposure_check(client.recordingManager.downloadImage(rel))

## 5 Acquire

Bias, ladder, reference. Runs in this cell (it asks you when the beta light has to move);
interrupting it (■) stops after the current frame.

In [ ]:
acq = slim.ProtocolAcquisition(run, client, detector, STEPS, shutter, operator=operator, env=env,
                               env_label=CAMERA, backend=preset["backend"])
try:
    acq.run()
except KeyboardInterrupt:
    acq.stop()
print(acq.state)

## 6 Finish

In [ ]:
env.stop()
run.save_json("camera_end.json", bp.parameters(client, detector)[1])
run.log("run finished")

## 7 Analysis

Reads only the run folder; point `RUN_DIR` at an older run to redraw it. `e_per_adu` goes into
S2 (S3 picks up the latest S1 run of each camera by itself). For the EMCCD it is per
photo-electron at EM gain 300, excess noise taken out.

In [ ]:
RUN_DIR = run.dir      # or e.g. Path("../data/S1_calibration/<run folder>")
summary = slim.redraw(RUN_DIR, show=True)
for key in ("e_per_adu", "adu_per_e", "read_noise_e", "offset_adu", "ptc_rel_residual_rms",
            "linearity_max_residual_pct", "dark_current_e_px_s", "ref_total_e_s", "warnings"):
    print(f"{key:28s} {summary.get(key)}")